# Notebook 6: Full Electrode-Wise Analysis (All 58 Electrodes)

Notebook 2 evaluated a subset of electrodes; this notebook evaluates **every one of the 58 electrodes individually** with all 8 models, using only the 20 spectral / band-power features per electrode.

Two protocols (identical to Notebooks 2 and 4):
1. **Within-session 80:20** – chronological 80% train / 20% test per user, equal-size impostor test set from other users' test portions.
2. **Multi-session LOSO** – train on 3 sessions, test on the 4th, balanced impostors.

Outputs: per-electrode ranking, per-region electrode ranking, best electrode per model, and a region-level aggregate of the single-electrode results.


In [1]:
# Environment setup, reproducible random seeds, and imports
import os
import sys
import time
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import numpy as np
import pandas as pd
from scipy import signal
import mne
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_curve
from sklearn.svm import SVC
from sklearn.ensemble import ExtraTreesClassifier
import xgboost as xgb
import lightgbm as lgb

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    _ = torch.zeros(1, device=DEVICE)

print(f"Compute Device: {DEVICE} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")
print(f"Random Seed   : {RANDOM_SEED}")


Compute Device: cuda (NVIDIA GeForce RTX 3050 Laptop GPU)
Random Seed   : 42


## Section 1: Spectral Feature Extraction (same pipeline as Notebooks 2-5)

In [2]:
# High-performance vectorized spectral feature extractor
_trapezoid = getattr(np, "trapezoid", getattr(np, "trapz", None))

FREQ_BANDS = {
    "delta": (1.0, 4.0),
    "theta": (4.0, 8.0),
    "alpha": (8.0, 13.0),
    "beta":  (13.0, 30.0),
    "gamma": (30.0, 50.0)
}

def extract_spectral_features_tensor(data, sfreq=250.0):
    """Vectorized extraction of 20 spectral features per electrode channel."""
    n_epochs, n_channels, n_times = data.shape
    freqs, psd = signal.welch(data, fs=sfreq, nperseg=min(256, n_times), axis=-1)
    
    # 1. Absolute band powers (5)
    bp_list = []
    for (f_lo, f_hi) in FREQ_BANDS.values():
        mask = (freqs >= f_lo) & (freqs <= f_hi)
        p = _trapezoid(psd[:, :, mask], freqs[mask], axis=-1)
        bp_list.append(p)
    bp_arr = np.stack(bp_list, axis=-1)
    
    # 2. Total power (1-50 Hz) (1)
    tot_mask = (freqs >= 1.0) & (freqs <= 50.0)
    total_power = np.expand_dims(_trapezoid(psd[:, :, tot_mask], freqs[tot_mask], axis=-1), axis=-1) + 1e-12
    
    # 3. Relative band powers (5)
    rel_bp = bp_arr / total_power
    
    # 4. Spectral band power ratios (4)
    delta = bp_arr[..., 0] + 1e-12
    theta = bp_arr[..., 1] + 1e-12
    alpha = bp_arr[..., 2] + 1e-12
    beta  = bp_arr[..., 3] + 1e-12
    tbr = np.expand_dims(theta / beta, -1)
    tar = np.expand_dims(theta / alpha, -1)
    abr = np.expand_dims(alpha / beta, -1)
    slow_fast = np.expand_dims((delta + theta) / (alpha + beta), -1)
    ratios = np.concatenate([tbr, tar, abr, slow_fast], axis=-1)
    
    # 5. Spectral distribution & shape metrics (5)
    freqs_sub = freqs[tot_mask]
    psd_sub = psd[:, :, tot_mask]
    p_sum = psd_sub.sum(axis=-1, keepdims=True) + 1e-12
    centroid = (psd_sub * freqs_sub).sum(axis=-1, keepdims=True) / p_sum
    spread = np.sqrt(((freqs_sub - centroid)**2 * psd_sub).sum(axis=-1, keepdims=True) / p_sum)
    
    cum_psd = np.cumsum(psd_sub, axis=-1)
    cum_norm = cum_psd / (cum_psd[..., -1:] + 1e-12)
    idx50 = (cum_norm >= 0.5).argmax(axis=-1, keepdims=True)
    idx95 = (cum_norm >= 0.95).argmax(axis=-1, keepdims=True)
    sef50 = np.take_along_axis(np.broadcast_to(freqs_sub, idx50.shape[:-1] + (len(freqs_sub),)), idx50, axis=-1)
    sef95 = np.take_along_axis(np.broadcast_to(freqs_sub, idx95.shape[:-1] + (len(freqs_sub),)), idx95, axis=-1)
    
    idx_peak = psd_sub.argmax(axis=-1, keepdims=True)
    peak_freq = np.take_along_axis(np.broadcast_to(freqs_sub, idx_peak.shape[:-1] + (len(freqs_sub),)), idx_peak, axis=-1)
    spectral_dist = np.concatenate([centroid, spread, sef50, sef95, peak_freq], axis=-1)
    
    features = np.concatenate([bp_arr, total_power, rel_bp, ratios, spectral_dist], axis=-1)
    return features.astype(np.float32)

dataset_path = Path("../preprocessed_dataset")
if not dataset_path.exists():
    dataset_path = Path("preprocessed_dataset")

fif_files = sorted(list(dataset_path.glob("**/*_desc-preproc_epo.fif")))
EPOCH_CAP = 45

sample_ep = mne.read_epochs(str(fif_files[0]), preload=False, verbose=False)
ALL_CHANNELS = list(sample_ep.ch_names)
CHANNEL_TO_IDX = {ch: i for i, ch in enumerate(ALL_CHANNELS)}

REGIONS = {
    "Frontal": ["Fp1", "Fpz", "Fp2", "AF3", "AF4", "F7", "F5", "F3", "F1", "Fz", "F2", "F4", "F6", "F8"],
    "Central": ["FC5", "FC3", "FC1", "FCz", "FC2", "FC4", "FC6", "C5", "C3", "C1", "Cz", "C2", "C4", "C6"],
    "Temporal": ["T7", "T8", "TP7", "TP8", "CP5", "CP6"],
    "Parietal": ["CP3", "CP1", "CPz", "CP2", "CP4", "P7", "P5", "P3", "P1", "Pz", "P2", "P4", "P6", "P8"],
    "Occipital": ["PO7", "PO5", "PO3", "POz", "PO4", "PO6", "PO8", "O1", "Oz", "O2"],
    "ALL": ALL_CHANNELS
}

print(f"Extracting spectral features for all {len(fif_files)} files (Capped at {EPOCH_CAP} epochs)...")
t0 = time.time()
session_data = {}

for fpath in fif_files:
    fname = fpath.name
    parts = fpath.parts
    sub = [p for p in parts if p.startswith("sub-")][0]
    ses = [p for p in parts if p.startswith("ses-")][0]
    
    ep = mne.read_epochs(str(fpath), preload=True, verbose=False)
    raw_data = ep.get_data()[:EPOCH_CAP]
    sfreq = ep.info['sfreq']
    
    feats = extract_spectral_features_tensor(raw_data, sfreq=sfreq)
    
    key = (sub, ses)
    if key not in session_data:
        session_data[key] = []
    session_data[key].append(feats)

session_cache = {}
for (sub, ses), feat_list in session_data.items():
    session_cache[(sub, ses)] = np.vstack(feat_list)

SUBJECTS = sorted(list(set(sub for (sub, ses) in session_cache.keys())))
SUBJECT_SESSIONS = {
    sub: sorted([ses for (s, ses) in session_cache.keys() if s == sub])
    for sub in SUBJECTS
}

print(f"Spectral features ready in {time.time() - t0:.2f} seconds.")
print(f"Total Subjects: {len(SUBJECTS)} | Sessions per subject: 4 | Samples per session: 90")


Extracting spectral features for all 40 files (Capped at 45 epochs)...


Spectral features ready in 7.88 seconds.
Total Subjects: 5 | Sessions per subject: 4 | Samples per session: 90


## Section 2: Metrics and the 8 Models (same configuration as Notebooks 2-5)

In [3]:
# Biometric Metric Calculator
def evaluate_biometric_predictions(y_true, y_scores, threshold=0.5):
    """Compute standard biometric verification metrics: EER, FAR, FRR, FP, FN, Accuracy."""
    y_true = np.asarray(y_true).astype(int)
    y_scores = np.asarray(y_scores).astype(float)
    
    n_gen = int(np.sum(y_true == 1))
    n_imp = int(np.sum(y_true == 0))
    
    fpr, tpr, thresholds = roc_curve(y_true, y_scores)
    fnr = 1.0 - tpr
    eer_idx = np.nanargmin(np.abs(fpr - fnr))
    eer = float((fpr[eer_idx] + fnr[eer_idx]) / 2.0)
    
    y_pred = (y_scores >= threshold).astype(int)
    tp = int(np.sum((y_true == 1) & (y_pred == 1)))
    tn = int(np.sum((y_true == 0) & (y_pred == 0)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1)))
    fn = int(np.sum((y_true == 1) & (y_pred == 0)))
    
    accuracy = float((tp + tn) / (n_gen + n_imp)) if (n_gen + n_imp) > 0 else 0.0
    far = float(fp / n_imp) if n_imp > 0 else 0.0
    frr = float(fn / n_gen) if n_gen > 0 else 0.0
    
    return {
        "Accuracy": accuracy,
        "EER": eer,
        "FAR": far,
        "FRR": frr,
        "FP": fp,
        "FN": fn
    }

# Deep Learning Architectures
class ResMLPNet(nn.Module):
    def __init__(self, in_features):
        super().__init__()
        self.fc1 = nn.Linear(in_features, 64)
        self.relu = nn.ReLU()
        self.res1 = nn.Linear(64, 64)
        self.drop = nn.Dropout(0.1)
        self.head = nn.Linear(64, 1)
    def forward(self, x):
        h = self.relu(self.fc1(x))
        h = h + self.drop(self.relu(self.res1(h)))
        return self.head(h).view(-1)

class CNN1DNet(nn.Module):
    def __init__(self, in_features):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=5, padding=2),
            nn.BatchNorm1d(16),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(4),
            nn.Flatten(),
            nn.Linear(32 * 4, 1)
        )
    def forward(self, x):
        x = x.unsqueeze(1)
        return self.net(x).view(-1)

class BiLSTMAttentionNet(nn.Module):
    def __init__(self, in_features, feat_dim=20):
        super().__init__()
        self.feat_dim = feat_dim
        dim = feat_dim if (in_features % feat_dim == 0 and in_features >= feat_dim) else in_features
        self.is_multichannel = (in_features % feat_dim == 0 and in_features >= feat_dim)
        self.lstm = nn.LSTM(dim if self.is_multichannel else 1, 16, batch_first=True, bidirectional=True)
        self.att = nn.Linear(32, 1)
        self.fc = nn.Linear(32, 1)
    def forward(self, x):
        if self.is_multichannel:
            x_seq = x.view(x.shape[0], -1, self.feat_dim)
        else:
            x_seq = x.unsqueeze(-1)
        out, _ = self.lstm(x_seq)
        weights = torch.softmax(self.att(out), dim=1)
        ctx = torch.sum(out * weights, dim=1)
        return self.fc(ctx).view(-1)

class SpecFormerNet(nn.Module):
    def __init__(self, in_features, d_model=24, nhead=3):
        super().__init__()
        if in_features % 20 == 0 and in_features >= 20:
            self.tok_proj = nn.Linear(20, d_model)
            self.multi_tok = True
        else:
            self.tok_proj = nn.Linear(1, d_model)
            self.multi_tok = False
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=48, batch_first=True, dropout=0.1)
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=1)
        self.fc = nn.Linear(d_model, 1)
    def forward(self, x):
        if self.multi_tok:
            x_seq = x.view(x.shape[0], -1, 20)
        else:
            x_seq = x.unsqueeze(-1)
        h = self.tok_proj(x_seq)
        enc = self.encoder(h)
        return self.fc(enc.mean(dim=1)).view(-1)

class PyTorchClassifierWrapper:
    def __init__(self, net_cls, epochs=15, lr=3e-3, random_state=42):
        self.net_cls = net_cls
        self.epochs = epochs
        self.lr = lr
        self.random_state = random_state
        self.model = None

    def fit(self, X, y):
        torch.manual_seed(self.random_state)
        in_dim = X.shape[1]
        self.model = self.net_cls(in_dim).to(DEVICE)
        self.model.train()
        optimizer = torch.optim.Adam(self.model.parameters(), lr=self.lr)
        criterion = nn.BCEWithLogitsLoss()
        
        X_t = torch.tensor(X, dtype=torch.float32, device=DEVICE)
        y_t = torch.tensor(y, dtype=torch.float32, device=DEVICE)
        
        for _ in range(self.epochs):
            optimizer.zero_grad()
            out = self.model(X_t)
            loss = criterion(out, y_t)
            loss.backward()
            optimizer.step()
        return self

    def predict_proba(self, X):
        self.model.eval()
        X_t = torch.tensor(X, dtype=torch.float32, device=DEVICE)
        with torch.no_grad():
            logits = self.model(X_t)
            probs = torch.sigmoid(logits).cpu().numpy().flatten()
        return np.column_stack([1.0 - probs, probs])

def get_model_pipeline(model_name, in_features):
    if model_name == "XGBoost":
        return xgb.XGBClassifier(
            n_estimators=60, max_depth=4, learning_rate=0.1,
            eval_metric="logloss", random_state=RANDOM_SEED, n_jobs=-1
        )
    elif model_name == "LightGBM":
        return lgb.LGBMClassifier(
            n_estimators=60, max_depth=4, num_leaves=15, learning_rate=0.1,
            random_state=RANDOM_SEED, n_jobs=-1, verbose=-1
        )
    elif model_name == "SVM_RBF":
        return SVC(C=10.0, kernel="rbf", gamma="scale", probability=True, random_state=RANDOM_SEED)
    elif model_name == "ExtraTrees":
        return ExtraTreesClassifier(n_estimators=60, max_depth=None, random_state=RANDOM_SEED, n_jobs=-1)
    elif model_name == "Res-MLP":
        return PyTorchClassifierWrapper(ResMLPNet, epochs=15, lr=3e-3, random_state=RANDOM_SEED)
    elif model_name == "1D-CNN":
        return PyTorchClassifierWrapper(CNN1DNet, epochs=15, lr=3e-3, random_state=RANDOM_SEED)
    elif model_name == "BiLSTM-Attention":
        return PyTorchClassifierWrapper(BiLSTMAttentionNet, epochs=15, lr=3e-3, random_state=RANDOM_SEED)
    elif model_name == "SpecFormer":
        return PyTorchClassifierWrapper(SpecFormerNet, epochs=15, lr=3e-3, random_state=RANDOM_SEED)
    else:
        raise ValueError(f"Unknown model: {model_name}")

ALL_MODEL_NAMES = [
    "XGBoost", "LightGBM", "SVM_RBF", "ExtraTrees",
    "Res-MLP", "1D-CNN", "BiLSTM-Attention", "SpecFormer"
]
print("All 8 advanced model architectures verified and registered.")


All 8 advanced model architectures verified and registered.


## Section 3: Electrode-to-Region Map and Per-Electrode Evaluation

Each electrode is tagged with its region from Notebook 1. For every electrode and model we run both protocols over all 5 users and average the metrics.


In [4]:
ELEC_REGION = {ch: r for r, chs in REGIONS.items() if r != 'ALL' for ch in chs}
assert set(ELEC_REGION) == set(ALL_CHANNELS) and len(ELEC_REGION) == 58

# Per-subject arrays in chronological session order (CE+OE per session), as in Notebook 2
SUB_ARR = {s: np.vstack([session_cache[(s, ses)] for ses in SUBJECT_SESSIONS[s]]) for s in SUBJECTS}  # (360, 58, 20)

def agg(metric_list):
    return {k: float(np.mean([m[k] for m in metric_list])) for k in ['Accuracy','EER','FAR','FRR','FP','FN']}

def within_session_for_electrode(ci):
    X = {s: SUB_ARR[s][:, ci, :] for s in SUBJECTS}
    res = {m: [] for m in ALL_MODEL_NAMES}; n_tr = n_g = n_i = 0
    for t in SUBJECTS:
        g = X[t]; ntr = int(len(g)*0.8)
        g_tr, g_te = g[:ntr], g[ntr:]
        others = [s for s in SUBJECTS if s != t]
        itr = ntr // len(others); ite = len(g_te) // len(others)
        i_tr = np.vstack([X[o][:int(len(X[o])*0.8)][:itr] for o in others])
        i_te = np.vstack([X[o][int(len(X[o])*0.8):][:ite] for o in others])
        Xtr = np.vstack([g_tr, i_tr]); ytr = np.r_[np.ones(len(g_tr)), np.zeros(len(i_tr))]
        Xte = np.vstack([g_te, i_te]); yte = np.r_[np.ones(len(g_te)), np.zeros(len(i_te))]
        sc = StandardScaler().fit(Xtr); Xtr, Xte = sc.transform(Xtr), sc.transform(Xte)
        n_tr, n_g, n_i = len(Xtr), len(g_te), len(i_te)
        for m in ALL_MODEL_NAMES:
            clf = get_model_pipeline(m, Xtr.shape[1]).fit(Xtr, ytr)
            res[m].append(evaluate_biometric_predictions(yte, clf.predict_proba(Xte)[:, 1]))
    return res, n_tr, n_g, n_i

LOSO = [([0,1,2],3), ([0,1,3],2), ([0,2,3],1), ([1,2,3],0)]
def loso_for_electrode(ci):
    D = {k: v[:, ci, :] for k, v in session_cache.items()}
    res = {m: [] for m in ALL_MODEL_NAMES}; n_tr = n_g = n_i = 0
    for tr_idx, te_idx in LOSO:
        for t in SUBJECTS:
            ss = SUBJECT_SESSIONS[t]
            g_tr = np.vstack([D[(t, ss[i])] for i in tr_idx]); g_te = D[(t, ss[te_idx])]
            others = [s for s in SUBJECTS if s != t]
            itr = len(g_tr) // len(others); ite = len(g_te) // len(others)
            i_tr = np.vstack([np.vstack([D[(o, SUBJECT_SESSIONS[o][i])] for i in tr_idx])[:itr] for o in others])
            i_te = np.vstack([D[(o, SUBJECT_SESSIONS[o][te_idx])][:ite] for o in others])
            Xtr = np.vstack([g_tr, i_tr]); ytr = np.r_[np.ones(len(g_tr)), np.zeros(len(i_tr))]
            Xte = np.vstack([g_te, i_te]); yte = np.r_[np.ones(len(g_te)), np.zeros(len(i_te))]
            sc = StandardScaler().fit(Xtr); Xtr, Xte = sc.transform(Xtr), sc.transform(Xte)
            n_tr, n_g, n_i = len(Xtr), len(g_te), len(i_te)
            for m in ALL_MODEL_NAMES:
                clf = get_model_pipeline(m, Xtr.shape[1]).fit(Xtr, ytr)
                res[m].append(evaluate_biometric_predictions(yte, clf.predict_proba(Xte)[:, 1]))
    return res, n_tr, n_g, n_i

rows_ws, rows_ls = [], []
t0 = time.time()
for k, ch in enumerate(ALL_CHANNELS):
    ci = CHANNEL_TO_IDX[ch]
    for rows, fn in ((rows_ws, within_session_for_electrode), (rows_ls, loso_for_electrode)):
        res, n_tr, n_g, n_i = fn(ci)
        for m, ml in res.items():
            rows.append({'Electrode': ch, 'Region': ELEC_REGION[ch], 'Model': m, 'N_train': n_tr,
                         'N_genuine_test': n_g, 'N_impostor_test': n_i, **agg(ml)})
    if (k + 1) % 10 == 0:
        print(f'{k+1}/58 electrodes done ({time.time()-t0:.0f}s)')

df_ws, df_ls = pd.DataFrame(rows_ws), pd.DataFrame(rows_ls)
print(f'Done: {len(df_ws)} within-session rows, {len(df_ls)} LOSO rows in {time.time()-t0:.0f}s')


10/58 electrodes done (110s)


20/58 electrodes done (219s)


30/58 electrodes done (327s)


40/58 electrodes done (434s)


50/58 electrodes done (542s)


Done: 464 within-session rows, 464 LOSO rows in 629s


## Section 4: Electrode Rankings (mean over the 8 models, and over the 4 ML models)

In [5]:
fmt = {'Accuracy':'{:.4f}','EER':'{:.4f}','FAR':'{:.4f}','FRR':'{:.4f}','FP':'{:.1f}','FN':'{:.1f}'}
cols = ['Accuracy','EER','FAR','FRR','FP','FN']
ML_MODELS = ['XGBoost','LightGBM','SVM_RBF','ExtraTrees']

def rank_table(df, models=None, top=15):
    d = df if models is None else df[df['Model'].isin(models)]
    return d.groupby(['Electrode','Region'])[cols].mean().reset_index().sort_values('EER').head(top).reset_index(drop=True)

print('TOP 15 ELECTRODES - WITHIN-SESSION 80:20 (mean over all 8 models)')
display(rank_table(df_ws).style.format(fmt))
print('TOP 15 ELECTRODES - MULTI-SESSION LOSO (mean over all 8 models)')
display(rank_table(df_ls).style.format(fmt))
print('BOTTOM 5 ELECTRODES - MULTI-SESSION LOSO')
full = df_ls.groupby(['Electrode','Region'])[cols].mean().reset_index().sort_values('EER')
display(full.tail(5).reset_index(drop=True).style.format(fmt))


TOP 15 ELECTRODES - WITHIN-SESSION 80:20 (mean over all 8 models)


,Electrode,Region,Accuracy,EER,FAR,FRR,FP,FN
0,P3,Parietal,0.8438,0.1528,0.1642,0.1483,11.8,10.7
1,C6,Central,0.8302,0.1618,0.2003,0.1392,14.4,10.0
2,CP5,Temporal,0.8394,0.1646,0.2149,0.1062,15.5,7.7
3,F2,Frontal,0.8271,0.1663,0.2052,0.1406,14.8,10.1
4,F4,Frontal,0.8309,0.1681,0.1924,0.1458,13.8,10.5
5,P1,Parietal,0.8196,0.1694,0.1899,0.1708,13.7,12.3
6,AF3,Frontal,0.8208,0.1707,0.1882,0.1701,13.6,12.2
7,PO3,Occipital,0.8165,0.1727,0.1806,0.1865,13.0,13.4
8,F1,Frontal,0.8172,0.1738,0.2212,0.1444,15.9,10.4
9,Fp1,Frontal,0.8161,0.1750,0.1684,0.1993,12.1,14.3


TOP 15 ELECTRODES - MULTI-SESSION LOSO (mean over all 8 models)


,Electrode,Region,Accuracy,EER,FAR,FRR,FP,FN
0,P8,Parietal,0.8352,0.1513,0.1761,0.1538,15.5,13.8
1,P4,Parietal,0.8366,0.1546,0.1726,0.1544,15.2,13.9
2,P6,Parietal,0.8296,0.1561,0.1862,0.1551,16.4,14.0
3,P2,Parietal,0.8282,0.1643,0.1776,0.1661,15.6,14.9
4,CP6,Temporal,0.8175,0.1676,0.2036,0.1619,17.9,14.6
5,Cz,Central,0.8243,0.1693,0.1803,0.1712,15.9,15.4
6,TP8,Temporal,0.8146,0.1700,0.2045,0.1667,18.0,15.0
7,Pz,Parietal,0.8263,0.1705,0.1763,0.1711,15.5,15.4
8,C5,Central,0.8143,0.1730,0.2017,0.1700,17.8,15.3
9,CP4,Parietal,0.8144,0.1765,0.1989,0.1726,17.5,15.5


BOTTOM 5 ELECTRODES - MULTI-SESSION LOSO


,Electrode,Region,Accuracy,EER,FAR,FRR,FP,FN
0,F8,Frontal,0.7717,0.2194,0.2269,0.2297,20.0,20.7
1,O1,Occipital,0.7698,0.2210,0.2290,0.2315,20.1,20.8
2,T8,Temporal,0.7727,0.2229,0.2258,0.2288,19.9,20.6
3,PO4,Occipital,0.7733,0.2251,0.2298,0.2236,20.2,20.1
4,Oz,Occipital,0.7489,0.2451,0.2450,0.2570,21.6,23.1


## Section 5: Region-Level Aggregation, Best Electrode per Model, and Region Comparison

In [6]:
for name, df in (('WITHIN-SESSION 80:20', df_ws), ('MULTI-SESSION LOSO', df_ls)):
    print(f'REGION AGGREGATE OF SINGLE-ELECTRODE RESULTS - {name} (mean over electrodes and 8 models)')
    reg = df.groupby('Region').agg(N_electrodes=('Electrode','nunique'), **{c:(c,'mean') for c in cols}).reset_index().sort_values('EER')
    display(reg.style.format({**fmt, 'N_electrodes':'{:d}'}))
    print(f'BEST ELECTRODE PER MODEL - {name}')
    idx = df.groupby('Model')['EER'].idxmin()
    display(df.loc[idx, ['Model','Electrode','Region','N_train','N_genuine_test','N_impostor_test']+cols].sort_values('EER').reset_index(drop=True).style.format(fmt))


REGION AGGREGATE OF SINGLE-ELECTRODE RESULTS - WITHIN-SESSION 80:20 (mean over electrodes and 8 models)


,Region,N_electrodes,Accuracy,EER,FAR,FRR,FP,FN
1,Frontal,14,0.8053,0.1863,0.2076,0.1817,14.9,13.1
3,Parietal,14,0.8037,0.1873,0.2116,0.1811,15.2,13.0
0,Central,14,0.7992,0.1959,0.2270,0.1747,16.3,12.6
4,Temporal,6,0.7822,0.2069,0.2297,0.2059,16.5,14.8
2,Occipital,10,0.7784,0.2180,0.2191,0.2241,15.8,16.1


BEST ELECTRODE PER MODEL - WITHIN-SESSION 80:20


,Model,Electrode,Region,N_train,N_genuine_test,N_impostor_test,Accuracy,EER,FAR,FRR,FP,FN
0,LightGBM,AF3,Frontal,576,72,72,0.8889,0.0944,0.1250,0.0972,9.0,7.0
1,XGBoost,P3,Parietal,576,72,72,0.8903,0.0958,0.1111,0.1083,8.0,7.8
2,ExtraTrees,AF3,Frontal,576,72,72,0.8944,0.1014,0.1167,0.0944,8.4,6.8
3,SVM_RBF,F4,Frontal,576,72,72,0.8778,0.1042,0.1472,0.0972,10.6,7.0
4,SpecFormer,Fz,Frontal,576,72,72,0.8167,0.1625,0.2444,0.1222,17.6,8.8
5,Res-MLP,P3,Parietal,576,72,72,0.8278,0.1722,0.2083,0.1361,15.0,9.8
6,1D-CNN,C6,Central,576,72,72,0.7889,0.1917,0.1944,0.2278,14.0,16.4
7,BiLSTM-Attention,C6,Central,576,72,72,0.7931,0.2056,0.2139,0.2000,15.4,14.4


REGION AGGREGATE OF SINGLE-ELECTRODE RESULTS - MULTI-SESSION LOSO (mean over electrodes and 8 models)


,Region,N_electrodes,Accuracy,EER,FAR,FRR,FP,FN
3,Parietal,14,0.8150,0.1773,0.1939,0.1764,17.1,15.9
4,Temporal,6,0.8041,0.1873,0.2116,0.1805,18.6,16.2
0,Central,14,0.8053,0.1892,0.2066,0.1831,18.2,16.5
1,Frontal,14,0.8020,0.1910,0.2074,0.1887,18.2,17.0
2,Occipital,10,0.7814,0.2112,0.2180,0.2192,19.2,19.7


BEST ELECTRODE PER MODEL - MULTI-SESSION LOSO


,Model,Electrode,Region,N_train,N_genuine_test,N_impostor_test,Accuracy,EER,FAR,FRR,FP,FN
0,SVM_RBF,Cz,Central,538,90,88,0.8798,0.1115,0.1136,0.1267,10.0,11.4
1,ExtraTrees,P2,Parietal,538,90,88,0.8654,0.1216,0.1545,0.1150,13.6,10.3
2,LightGBM,P8,Parietal,538,90,88,0.8643,0.1216,0.1449,0.1267,12.8,11.4
3,XGBoost,F2,Frontal,538,90,88,0.8483,0.1278,0.1653,0.1383,14.6,12.4
4,SpecFormer,P8,Parietal,538,90,88,0.8430,0.1466,0.1585,0.1556,13.9,14.0
5,Res-MLP,P6,Parietal,538,90,88,0.8171,0.1587,0.2028,0.1633,17.9,14.7
6,BiLSTM-Attention,P6,Parietal,538,90,88,0.7935,0.1930,0.2250,0.1883,19.8,16.9
7,1D-CNN,P8,Parietal,538,90,88,0.7711,0.2070,0.2528,0.2056,22.2,18.5


## Section 6: Electrode-by-Model EER Matrix (LOSO) for the 15 Best Electrodes

In [7]:
top = rank_table(df_ls, top=15)['Electrode'].tolist()
mat = df_ls[df_ls['Electrode'].isin(top)].pivot(index='Electrode', columns='Model', values='EER').loc[top]
display(mat.style.format('{:.3f}').background_gradient(cmap='RdYlGn_r', axis=None))
b = df_ls.loc[df_ls['EER'].idxmin()]
print(f"Best single electrode + model (LOSO): {b['Electrode']} ({b['Region']}) with {b['Model']}  EER={b['EER']:.4f}  Acc={b['Accuracy']:.4f}  FAR={b['FAR']:.4f}  FRR={b['FRR']:.4f}")


Model,1D-CNN,BiLSTM-Attention,ExtraTrees,LightGBM,Res-MLP,SVM_RBF,SpecFormer,XGBoost
Electrode,,,,,,,,
P8,0.207,0.194,0.124,0.122,0.164,0.122,0.147,0.130
P4,0.210,0.199,0.129,0.124,0.160,0.133,0.150,0.132
P6,0.210,0.193,0.137,0.126,0.159,0.138,0.150,0.137
P2,0.230,0.208,0.122,0.140,0.165,0.144,0.168,0.138
CP6,0.228,0.204,0.131,0.146,0.182,0.139,0.170,0.140
Cz,0.247,0.232,0.127,0.132,0.187,0.112,0.181,0.137
TP8,0.232,0.224,0.134,0.138,0.193,0.140,0.164,0.134
Pz,0.228,0.230,0.135,0.135,0.179,0.134,0.190,0.134
C5,0.229,0.222,0.147,0.137,0.187,0.155,0.172,0.133


Best single electrode + model (LOSO): Cz (Central) with SVM_RBF  EER=0.1115  Acc=0.8798  FAR=0.1136  FRR=0.1267
